# 05. Delay-SNN 分支训练（FP32）

第三个、也是结构最简单的分支：**Delay-SNN**，复现 DASIP 2024 论文
（Scrugli et al., "sEMG-Based Gesture Recognition with Spiking Neural Networks on
Low-Power FPGA"）的延迟编码 SNN 方法。项目位置和 Context/Hybrid 不同，在一个独立的
项目目录里：`training/semg_snn_fpga_reproduction/`——它比 Context/Hybrid
更早完成，是后续两个分支做硬件适配时参照的模板。

## 和 Context/Hybrid 的关键差异

| | Context / Hybrid | Delay-SNN |
|---|---|---|
| 输入表示 | 336 维统计特征 / 16 通道原始波形 | 96 维二值 delta 调制脉冲(16通道×3阶导数×2极性) |
| 网络结构 | 512/384 宽的 Linear+LIF | 96→64→128→64→13 的纯 Dense LIF,无卷积无注意力 |
| 衰减系数 | 可学习(PLIF) | **固定** decay=0.9,threshold=1.0 |
| 训练目标 | 交叉熵(分类) | **脉冲发放率回归**(MSE,目标类 0.2,非目标类 0.03) |
| 特色结构 | LayerNorm、GELU、Jaccard 注意力 | 前三层各带一个**可学习轴突延迟**(0~62 步) |
| checkpoint 选择依据 | 验证集 accuracy | 验证集 **loss** |

这些差异不是偶然的——它们直接对应论文原始方法的设计，这里如实复现，不做简化。

In [ ]:
import sys, random, json
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score

DELAY_ROOT = Path("training/semg_snn_fpga_reproduction")
sys.path.insert(0, str(DELAY_ROOT))

from model import PaperSNN, PaperSNNWithDelays, spike_rate_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NB_RUNS = DELAY_ROOT / "runs_notebook"
NB_RUNS.mkdir(exist_ok=True)

## 1. 数据：delta 调制编码

这个分支的预处理和 Context/Hybrid 完全不同（在 `prepare_db5.py` 里，这里不重复实现，
直接读取已经生成好的 `.npz`）：每个 16 通道原始波形样本、其一阶导数、二阶导数各自
过一次阈值 15 的 delta 调制（正负极性分开记为两路二值事件），拼成
`16 通道 × 3 阶 × 2 极性 = 96` 维的二值脉冲输入,100 个时间步(0.5秒窗口)。
train/val/test 的 repetition 划分和 Context/Hybrid 一致(1,2,4,6 / 3 / 5)，
所以三个分支在同一个测试窗口集合上是可比的。

In [ ]:
class NPZDataset(Dataset):
    def __init__(self, path: Path) -> None:
        data = np.load(path)
        self.x = data["x"]
        self.y = data["y"].astype(np.int64)
        self.subject = data["subject"]
        self.start = data["start"]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index: int):
        return (
            torch.from_numpy(self.x[index].astype(np.float32)),
            torch.tensor(self.y[index], dtype=torch.long),
            torch.tensor(self.subject[index], dtype=torch.long),
            torch.tensor(self.start[index], dtype=torch.long),
        )

data_dir = DELAY_ROOT / "data" / "processed"
train_set = NPZDataset(data_dir / "train.npz")
val_set = NPZDataset(data_dir / "val.npz")
test_set = NPZDataset(data_dir / "test.npz")
print(f"train={len(train_set)}  val={len(val_set)}  test={len(test_set)}  "
      f"input shape={tuple(train_set[0][0].shape)}")

## 2. 训练目标：脉冲发放率回归，不是交叉熵

`spike_rate_loss`（`model.py`）不是分类交叉熵，而是让**输出层每个神经元的平均发放率**
去逼近一个目标值：正确类别的神经元目标发放率 0.2，其余神经元目标发放率 0.03，
用 MSE 衡量。这是 SLAYER 类 SNN 训练常见的做法——训练信号直接作用在"这个神经元该
以多高频率发放"上，而不是在某个读出层做 softmax。预测时看的是"哪个输出神经元这个
窗口发放次数最多"（`output.sum(dim=1).argmax(dim=1)`），不需要额外的读出层。

In [ ]:
@torch.no_grad()
def evaluate_delay(model, loader, max_batches=0):
    model.eval()
    losses, predictions, targets = [], [], []
    layer_rates = []
    for batch_index, (x, y, _, _) in enumerate(loader):
        if max_batches and batch_index >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        output, rates = model(x)
        loss = spike_rate_loss(output, y)
        losses.append(float(loss))
        predictions.extend(output.sum(dim=1).argmax(dim=1).cpu().tolist())
        targets.extend(y.cpu().tolist())
        layer_rates.append([float(r) for r in rates])
    y_arr, p_arr = np.asarray(targets), np.asarray(predictions)
    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(y_arr, p_arr),
        "macro_f1": f1_score(y_arr, p_arr, average="macro"),
        "gesture_accuracy": float(np.mean(p_arr[y_arr != 0] == y_arr[y_arr != 0])),
        "layer_spike_rates": np.mean(layer_rates, axis=0).tolist(),
    }


def train_delay(model, run_name, epochs, lr, patience, batch_size=32, seed=42):
    """训练循环,和 train.py 的 main() 逻辑一致:按验证集 loss(不是 accuracy)选 checkpoint。"""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    loaders = {
        "train": DataLoader(train_set, batch_size, shuffle=True, num_workers=4,
                             pin_memory=True, generator=generator),
        "val": DataLoader(val_set, batch_size, num_workers=4, pin_memory=True),
        "test": DataLoader(test_set, batch_size, num_workers=4, pin_memory=True),
    }
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    run_dir = NB_RUNS / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_loss, stale = float("inf"), 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for x, y, _, _ in loaders["train"]:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            output, _ = model(x)
            loss = spike_rate_loss(output, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(float(loss.detach()))
        val_metrics = evaluate_delay(model, loaders["val"])
        print(f"epoch {epoch:02d}  train_loss={np.mean(losses):.4f}  "
              f"val_loss={val_metrics['loss']:.4f}  val_acc={val_metrics['accuracy']:.4f}")

        if val_metrics["loss"] < best_loss:
            best_loss, stale = val_metrics["loss"], 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "validation": val_metrics},
                       run_dir / "best.pt")
        else:
            stale += 1
            if stale >= patience:
                print("early stopping"); break

    ckpt = torch.load(run_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    test_metrics = evaluate_delay(model, loaders["test"])
    print(f"\n=== {run_name} 最终结果（第 {ckpt['epoch']} 轮,按验证集 loss 选中)===")
    print(f"test: accuracy={test_metrics['accuracy']:.4f}  macro_f1={test_metrics['macro_f1']:.4f}  "
          f"gesture_accuracy={test_metrics['gesture_accuracy']:.4f}")
    return run_dir / "best.pt", test_metrics

## 3. Stage 1：无延迟基线

先训练一个不带轴突延迟的普通 4 层 Dense LIF（`PaperSNN`）。真实项目里这一步跑了
30 个 epoch（在 100 epoch 预算内，验证 loss 进入平台期后手动停止，`patience=10`
还没触发），选中第 28 轮。这里为了让 notebook 首次跑起来更快，先示范用较小的
`epochs`；如果要复现论文数值，把 `epochs` 调到 30 附近、`patience=10`。

In [ ]:
baseline_model = PaperSNN(decay=0.9, threshold=1.0).to(device)
baseline_checkpoint, baseline_test_metrics = train_delay(
    baseline_model, run_name="delay_baseline_nb", epochs=30, lr=1e-3, patience=10,
)
print("\n参考值(真实项目 full_baseline, RESULTS.md): "
      "accuracy=0.8339 macro_f1=0.6322 (无延迟)")

## 4. Stage 2：加入可学习轴突延迟，热启动微调

`LearnableAxonalDelay` 给前三层的每个输出神经元一个可学习的延迟量（0~62 个时间步，
通过 sigmoid 参数化 + 线性插值实现可微分）。热启动自 stage 1 的权重，训练 15 个
epoch（真实项目 `delay62_finetune` 的确切配置）。这一步让测试准确率从 83.39%
提升到 **83.93%**——这就是本课程其它 notebook 里反复提到的 "Delay-SNN 83.93%" 数字的来源。

In [ ]:
delay_model = PaperSNNWithDelays(decay=0.9, threshold=1.0, max_delay=62, initial_delay=1.0).to(device)
init_state = torch.load(baseline_checkpoint, map_location=device, weights_only=False)["model"]
missing, unexpected = delay_model.load_state_dict(init_state, strict=False)
print(f"warm-start from {baseline_checkpoint}: missing={missing}  unexpected={unexpected}")

delay_checkpoint, delay_test_metrics = train_delay(
    delay_model, run_name="delay62_finetune_nb", epochs=15, lr=1e-3, patience=10,
)
print("\n参考值(真实项目 delay62_finetune, RESULTS.md): accuracy=0.8393 macro_f1=0.6561")

print("\n各层学到的延迟统计(时间步):")
for index, stats in enumerate(delay_model.delay_statistics(), start=1):
    print(f"  Layer {index}: min={stats['minimum']:.2f}  mean={stats['mean']:.2f}  max={stats['maximum']:.2f}")

延迟统计应该能看到一个模式（真实项目的结果）：第一层延迟很小（0.7~2.7 步），
第三层延迟明显更大、方差也更大（0.35~33.5 步）——网络学到了"越靠后的层，越需要
更长/更灵活的时间对齐"，这是轴突延迟这个机制真正在发挥作用的证据，而不是一个
训练完全没学到东西的死参数。

## 下一步

打开 [06_delay_quantization.ipynb](06_delay_quantization.ipynb)。这个分支的量化方式
和 Context/Hybrid（QAT，需要重新训练）会很不一样——它用的是更简单的**训练后量化
（PTQ）**，不需要重新训练，这本身就是一个值得对比的设计决策。